# 7. Exportación de las bases con llave de cruce

**Objetivo:** devolver BANCI y RNPDNO **idénticos a sus archivos de origen** —mismas filas,
mismas columnas, mismo formato, mismo nombre de archivo— con un bloque de columnas nuevas al
final que permita cruzarlos entre sí y saber **cómo** se hizo cada cruce.

Cada base sale en el formato en que entró: BANCI es un Excel de varias hojas y sale como
Excel; el RNPDNO sale como Parquet si entró como Parquet, y como Excel de tres hojas si entró
como Excel (`config.RNPDNO_LOCAL_FILE`, ver notebook 1).

| Columna | Para qué |
|---|---|
| `id_cruce` | **la llave**: el FUB del RNPDNO pareado. Vacío = sin cruce |
| `metodo_match` | `manual` · `estandar` · `refinado` (el ganador de la cascada) |
| `submetodo_match` | `match_fub` · `match_curp` · `match_nombre` · `definitivo` · … |
| `nivel_match` | 1 · 2 · 3 — el embudo del motor estándar |
| `estatus_match` | `con_cruce` · `candidato_revision_manual` · `sin_cruce` |

El diccionario completo está en [`docs/7_llaves_de_cruce.md`](../docs/7_llaves_de_cruce.md).

Los matches salen del universo que se construye en la sección 2: la exportación no vuelve
a correr los motores, toma la CAPA A de ese mismo universo (del caché, si sigue valiendo).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

from src.settings import Settings
from src.pipeline import Universo
from src.pipeline.exportacion_llaves import (
    exportar_bases_con_llave, COLUMNAS_BANCI, COLUMNAS_RNPDNO,
)

## 1. Configuración

El perfil decide **qué se cruza**, y hay dos:

| Perfil | Universo | Para qué |
|---|---|---|
| `reporte_diario` | desaparecidos del sexenio, sin localizados | los indicadores y las PPT |
| `cruce_total` | **todo BANCI contra todo el RNPDNO**, sin filtro de estatus ni de fecha | encontrar cada pareja posible |

Cada uno tiene su propia llave de caché y exporta a su propia carpeta
(`output/bases_con_llave/<perfil>/`), así que las dos corridas coexisten sin pisarse.

> ⚠️ `cruce_total` produce más matches y **algunos serán espurios**: quita a propósito
> los filtros que evitaban unir a un localizado con un desaparecido. No lo uses para
> los indicadores.

`clave_matching` viaja en la columna `version_match` de los archivos exportados:
identifica la corrida exacta que produjo cada llave.

In [ ]:
PERFIL = "reporte_diario"     # o "cruce_total" para el cruce completo

cfg = Settings.desde_perfil(PERFIL)
print(cfg.resumen())

## 2. Universo

Si el caché ya está, esto tarda segundos. Si no, ejecuta los tres motores (minutos).

In [ ]:
u = Universo.construir(cfg)
print()
print(u.resumen().to_string(index=False))

## 3. Exportación

Escribe los dos archivos en `output/bases_con_llave/` y **valida que el contenido original se
haya conservado**: BANCI celda por celda, el RNPDNO columna por columna (incluido el dtype).
Si algo no cuadra, falla aquí y no entrega nada: una diferencia silenciosa haría que todo lo
que se cruce después estuviera mal sin que nadie se enterara.

El RNPDNO son 409,241 filas × 62 columnas; se escribe y se vuelve a leer para compararlo.
Con `validar=False` se salta la comprobación (no recomendado para una entrega).

In [ ]:
res = exportar_bases_con_llave(u)

## 4. Qué quedó

Además de las dos bases, se escribe la **tabla puente** (`mapa_cruce_<corte>.parquet` y su
CSV): sólo los registros con cruce. Es lo más cómodo para usar desde R o SQL cuando sólo se
necesita el apareo y no las bases completas.

In [ ]:
print("BANCI     :", res.banci)
print("RNPDNO    :", res.rnpdno)
print("Puente    :", res.mapa)
print("Manifiesto:", res.manifiesto)
print()

puente = pd.read_parquet(res.mapa)
print(f"{len(puente):,} pares BANCI ↔ RNPDNO")
puente.head()

## 5. Cómo se usa el cruce

```python
ba = pd.read_excel(res.banci, sheet_name="Victimas")
rn = pd.read_parquet(res.rnpdno)

pares = ba[ba["id_cruce"].notna()].merge(
    rn, on="id_cruce", how="inner", suffixes=("_banci", "_rnpdno")
)
```

El RNPDNO en Parquet no viene separado en hojas: es una tabla única con las tres apiladas,
y `estatus` distingue a los localizados de los no localizados. Si entró como Excel, sale con
sus tres hojas: `rn = pd.concat(pd.read_excel(res.rnpdno, sheet_name=None).values())`.

> ⚠️ La relación es **N:1**: varias carpetas de BANCI pueden apuntar a la misma persona
> (4,663 FUB en la corrida vigente). Al cruzar de RNPDNO hacia BANCI se multiplican filas.
> `n_registros_banci` dice cuántas esperar, y `id_registro_banci_principal` apunta al
> registro representativo si se quiere un 1:1.

In [ ]:
# Desglose de cómo se resolvieron los cruces.
res.resumen

In [ ]:
# Contrato de columnas nuevas (el mismo que exige la validación).
print("BANCI :", ", ".join(COLUMNAS_BANCI))
print()
print("RNPDNO:", ", ".join(COLUMNAS_RNPDNO))

## 6. Base para la revisión del Secretariado

Los pares que un motor sostiene como match pero cuya **fecha de hechos no lo confirma**, para
que el Secretariado diga si valen (columna `VALIDO`: SI / NO). Es la base que antes se armaba
a mano con las salidas del script de R y de `normalizar_folios_v6.py`:

| Motor | Qué pares entran | Regla de origen |
|---|---|---|
| estándar | `match_definitivo == "DIFIERE"`: más de 10 días entre las dos fechas | script de R |
| refinado | definitivos cuya fecha no cupo en ±10 días o que no la tienen | `normalizar_folios_v6.py` |

`motor_origen` dice de cuál viene cada par (`estandar, refinado` si los dos lo proponen). No se
vuelve a mandar lo que el Secretariado ya contestó ni los pares cuyo FUB el motor manual no
podría aplicar. La hoja `resumen` dice cuántos pares propuso cada motor y cuántos se
descontaron. Detalle en `docs/7_llaves_de_cruce.md` §7.8.

> No cambia ningún resultado: sólo lee la capa A del universo y escribe
> `output/revision_sesnsp/<perfil>/revision_fecha_hechos_<corte>.xlsx`.

In [ ]:
from src.pipeline.revision_sesnsp import exportar_revision_fecha_hechos

revision = exportar_revision_fecha_hechos(u)
revision.base.head()

## 7. Las dos corridas, lado a lado

Cifras del corte del 7 de septiembre de 2026.

| | `reporte_diario` | `cruce_total` |
|---|---:|---:|
| Carpetas de BANCI con cruce | 7,253 | **25,438** |
| Personas del RNPDNO con cruce | 7,171 | **24,899** |
| Carpetas sin cruce | 23,762 | 732 |
| Candidatos a revisión manual | 3,762 | 8,607 |
| Tiempo del matching | ~2.4 min | **~13.4 min** |

El cruce total tardaba más de 100 horas antes de optimizar el motor refinado
(ver `docs/3_metodologia_de_matching.md`, §3.10).

In [ ]:
# Compara las dos corridas. Ambas leen del caché si ya se construyeron; si no, el
# perfil que falte corre aquí sus motores (`cruce_total`: más de diez minutos).
import pandas as pd

from src.slices import Slice

filas = []
for nombre in ("reporte_diario", "cruce_total"):
    c = Settings.desde_perfil(nombre)
    uu = Universo.construir(c, verbose=False)
    filas.append({
        "perfil": nombre,
        "RNPDNO con cruce": uu.contar(Slice("rn-si-ba", "min-max")),
        "BANCI con cruce": uu.contar(Slice("ba-si-rn", "min-max")),
        "BANCI sin cruce": uu.contar(Slice("ba-no-rn", "min-max")),
    })
pd.DataFrame(filas)